# exp_021 · Level 1 — VAE Latent Space (`z_t`)

Analyses the **noisy video latent trajectory** recorded during Stage-1 denoising.

| Tensor | Shape | What it encodes |
|--------|-------|-----------------|
| `z_t[τ, :, p, :, :]` | `[S=40, C=128, F'=16, H'=16, W'=24]` | Noisy video latent at denoising step τ, latent frame p |
| `z_final` | `[C=128, F'=16, H'=16, W'=24]` | Fully denoised clean latent z₀ |

---

### Conditioning geometry

```
Latent frames:  p=0  p=1  p=2  p=3  |  p=4  p=5 ... p=11  |  p=12  p=13  p=14  p=15
                ←  start-clip (cyan) →  ← free middle (white) →  ← end-clip (magenta) →
                  K_LAT=4 frames          8 model-generated frames    K_LAT=4 frames
```

### Features extracted (each aggregates over C=128 channels and H'×W'=384 spatial positions)

| Feature | Computation | What it measures |
|---------|-------------|-----------------|
| **`norm_z[τ, p]`** | `‖z_t[τ, :, p, :, :]‖₂` | Noise level proxy. Decreases from τ=0 → τ=39. Conditioned frames should diverge from free-middle early. |
| **`speed_z[τ, p]`** | `‖z_t[τ,:,p+1,:,:] − z_t[τ,:,p,:,:]‖₂` | Spatial displacement between adjacent latent frames at step τ. Large = large content difference. |
| **`curvature_z[τ, p]`** | `‖z_t(p+2) − 2z_t(p+1) + z_t(p)‖₂` | **Primary dissolve signal.** Measures how sharply the frame trajectory bends at p. A spike = hard transition. |
| **`angular_z[τ, p]`** | `cos(Δz_t(p), Δz_t(p+1))` | Direction consistency. cos < 0 = trajectory reverses = hard content change. |
| **`step_size_z[τ, p]`** | `‖z_t[τ+1,p] − z_t[τ,p]‖₂` | How much frame p changes per denoising step. Frames with high late-step changes are uncertain. |

> **Shading:** 🔵 Cyan = start-clip `p=0..3` · 🟣 Magenta = end-clip `p=12..15` · ⬜ White = free-middle `p=4..11`  
> **Green solid line** = ground-truth transition annotation (from manual labelling).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # find trajectory_utils.py

from trajectory_utils import *

# Load all 10 samples (extracts features, frees raw tensors)
records = load_all_records()

# Load manual GT transition annotations
gt_dict = load_gt_annotations()

n_gt = sum(1 for v in gt_dict.values() if v is not None)
print(f"\n✓ GT annotations available for {n_gt}/{len(gt_dict)} samples")
for sid, gs in sorted(gt_dict.items()):
    gp_str = f"  p≈{gt_latent(sid, gt_dict):.1f}" if gs is not None else "  —"
    gs_str = f"{gs:.2f}s" if gs is not None else " —"
    print(f"  {sid:<50s}  {gs_str:>7s}{gp_str}")


## Video Gallery — All 10 Generated Clips

Generated clips sorted by semantic class (border colour = class colour).

**Watch each video first** — note where you see the visual cut or dissolve,
then compare with the latent signals in the plots below.

**Semantic class key:**
- 🔵 **C1**: similar context / similar category / similar motion (easiest — expect smooth transition)
- 🟢 **C2**: similar context / similar category / different motion
- 🟠 **C5**: different context / similar category / similar motion
- 🔴 **C6**: different context / similar category / different motion
- 🟣 **C8**: different context / different category / different motion (hardest — expect sharp cut)


In [ ]:
display(video_grid(records, cols=5, width=300))


---
## 1.1 — VAE Latent Feature Heatmaps (`τ × p`)

Each row in the grid below is one sample; each column is one feature.
The heatmap axes are denoising step τ (y, 0=noisy) vs. latent frame p (x).

**What to look for:**

| Feature | Dissolve signature |
|---------|--------------------|
| `curvature_z` | Vertical bright stripe at the transition frame — present across all 40 steps for hard cuts |
| `angular_z` | Blue vertical band (cos < 0) at the cut frame — the trajectory reverses direction |
| `pred_mag` | Persistent bright stripe from earliest steps — the velocity is always large at the transition frame |
| `norm_z` | Conditioned frames (cyan/magenta) should stand out from free-middle early in denoising |

**Compare across classes:** Do class 8 samples show a sharper curvature stripe than class 1?


In [ ]:
import matplotlib.gridspec as gridspec

n = len(records)
fig = plt.figure(figsize=(18, n * 2.4 + 1.0))
fig.suptitle(
    "Level 1 — VAE Latent Features  (τ × p)  |  cyan=start  magenta=end  green line=GT",
    fontsize=12, fontweight="bold", y=0.995
)

panels = [
    ("norm_z",      "‖z_t(p)‖  noise level",          "PuBu_r",   False),
    ("speed_z",     "speed_z  ‖Δ_p z_t‖",              "YlOrRd",   False),
    ("curvature_z", "curvature_z  ‖Δ²_p z_t‖ ◄DISSOLVE", "hot",  False),
    ("angular_z",   "angular_z  cos(Δ,Δ) ◄FLIP",       "RdYlGn",   True),
    ("pred_mag",    "pred_mag  ‖v_θ(p)‖",               "plasma",   False),
    ("step_size_z", "step_size  ‖Δ_τ z_t(p)‖",         "magma",    False),
]

n_cols = len(panels)
gs_grid = gridspec.GridSpec(n, n_cols, figure=fig, hspace=0.55, wspace=0.38)

for row, r in enumerate(records):
    f = r["feats"]
    F = f["F"]
    for col, (key, title, cmap, div) in enumerate(panels):
        ax = fig.add_subplot(gs_grid[row, col])
        heatmap(ax, f[key], title if row == 0 else "", cmap, div, F=F)
        if col == 0:
            ax.set_ylabel(f"{r['short_id']}\nstep τ", fontsize=6.5, labelpad=3)

plt.tight_layout(rect=[0, 0, 1, 0.995])
plt.show()
print("Row = one sample.  Column = one feature.  Compare curvature_z stripe across classes.")


---
## 1.2 — Clean Latent `z₀` Signals Per Sample

`z₀ = z_t[τ=39]` is the fully denoised latent — the direct precursor to the decoded video.
Its geometry is the most diagnostic for locating the dissolve frame.

**Four panels per sample:**
1. **`‖z₀(p)‖` latent energy** — frame energy profile
2. **`‖Δ²z₀‖` curvature** — where the latent trajectory bends most sharply
3. **`cos(Δ,Δ)` angular consistency** — where the trajectory reverses direction (red fill = reversal zone)
4. **`step_size_z` heatmap** — how much each frame changes at each denoising step (late = uncertain)

**Green solid line** = ground-truth transition (seconds converted to latent frame).  
If the curvature peak and angular minimum align with the GT line → the geometric signal correctly identifies the transition.


In [ ]:
n = len(records)
fig, axes = plt.subplots(n, 4, figsize=(18, n * 2.0))
fig.suptitle(
    "Level 1 — Clean Latent z₀ Signals  |  cyan=start  magenta=end  🟢=GT annotation",
    fontsize=11, fontweight="bold"
)

for row, r in enumerate(records):
    f      = r["feats"]
    F      = f["F"]
    frames = np.arange(F)
    f_c2   = np.arange(F - 2) + 1.0
    col    = r["color"]
    sid    = r["sample_id"]

    for colidx in range(4):
        ax = axes[row, colidx]
        if colidx == 0:
            ax.bar(frames, f["norm_z0"], width=0.75, color=col, alpha=0.8)
            add_gt_vline(ax, sid, gt_dict, label=(row == 0))
            shade_cond(ax, F, F=F)
            ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
            ax.tick_params(labelsize=6)
            if row == 0: ax.set_title("‖z₀(p)‖  latent energy", fontsize=8)
            ax.set_ylabel(r["short_id"], fontsize=6, labelpad=2)

        elif colidx == 1:
            ax.bar(f_c2, f["curvature_z0"], width=0.75, color="#FF5722", alpha=0.85)
            add_gt_vline(ax, sid, gt_dict, label=(row == 0))
            shade_cond(ax, F-2, F=F)
            ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
            ax.tick_params(labelsize=6)
            if row == 0: ax.set_title("‖Δ²z₀‖  curvature  ◄ DISSOLVE", fontsize=8)

        elif colidx == 2:
            ang = f["angular_z0"]
            ax.plot(f_c2, ang, "o-", color="#2196F3", lw=1.5, ms=3.5)
            ax.axhline(0, color="gray", lw=0.8, ls=":")
            ax.fill_between(f_c2, ang, 0, where=(ang < 0), color="red", alpha=0.3)
            add_gt_vline(ax, sid, gt_dict, label=(row == 0))
            shade_cond(ax, F-2, F=F)
            ax.set_ylim(-1.15, 1.15)
            ax.set_xlim(-0.5, F-0.5); ax.set_xticks(range(F))
            ax.tick_params(labelsize=6)
            min_lbl = f"min={f['angular_z0'].min():.3f}"
            if row == 0: ax.set_title(f"cos(Δ,Δ)  direction consistency\n{min_lbl}", fontsize=8)
            else: ax.set_title(min_lbl, fontsize=7)

        else:
            ax.imshow(f["step_size_z"].T, aspect="auto", cmap="magma",
                      origin="lower", extent=[-0.5, f["S"]-0.5, -0.5, F-0.5])
            gp = gt_latent(sid, gt_dict)
            if gp is not None:
                ax.axhline(gp, color="#4CAF50", lw=1.5, ls="-", alpha=0.9)
            ax.set_xlabel("step τ", fontsize=6); ax.set_ylabel("frame p", fontsize=6)
            ax.tick_params(labelsize=6)
            if row == 0:
                ax.set_title("step_size_z  ‖Δ_τ z_t(p)‖\n(when does frame change?)", fontsize=7)

plt.tight_layout()
plt.show()


---
## 1.3 — Video + z₀ Signal Comparison (Inline)

Side-by-side: the generated video alongside three key 1-D signals from `z₀`.

**How to use this view:**
1. Play the video and note the perceptual transition time (seconds).
2. Convert: `p ≈ time_s × 24 / 8`.
3. Check if the curvature peak and angular minimum align with the green GT line.

All signals are from the **final clean latent** (`τ=39`).  
🟢 Green vertical line = ground-truth transition annotation.


In [ ]:
import io

all_html = []
all_html.append(
    '<h3 style="font-family:sans-serif">Level 1 — Video + z₀ Signals Per Sample</h3>'
    '<p style="font-family:sans-serif;font-size:12px">'
    'Each row: 🎬 video  |  curvature  |  angular consistency  |  velocity magnitude<br>'
    '🟢 solid = GT annotation</p>'
)

for r in records:
    f      = r["feats"]
    F      = f["F"]
    f_c2   = np.arange(F - 2) + 1.0
    frames = np.arange(F)
    ang    = f["angular_z0"]
    col    = r["color"]
    sid    = r["sample_id"]
    gs     = gt_dict.get(sid)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 2.4))
    gt_str = f"  |  GT={gs:.2f}s" if gs is not None else ""
    fig.suptitle(
        r["short_id"] + f"  |  Class {r['cls']}: {r['cls_label']}" + gt_str,
        fontsize=9, fontweight="bold", color=col
    )

    # curvature
    ax1.bar(f_c2, f["curvature_z0"], width=0.75, color="#FF5722", alpha=0.85)
    add_gt_vline(ax1, sid, gt_dict, label=True)
    shade_cond(ax1, F-2, F=F)
    ax1.set_xlim(-0.5, F-0.5); ax1.set_xticks(range(F))
    ax1.set_title(f"curvature_z0", fontsize=8)
    ax1.set_xlabel("frame p"); ax1.legend(fontsize=7)
    ax1.tick_params(labelsize=7)

    # angular
    ax2.plot(f_c2, ang, "o-", color="#2196F3", lw=1.5, ms=3.5)
    ax2.axhline(0, color="gray", lw=0.8, ls=":")
    ax2.fill_between(f_c2, ang, 0, where=(ang < 0), color="red", alpha=0.3)
    add_gt_vline(ax2, sid, gt_dict, label=False)
    shade_cond(ax2, F-2, F=F)
    ax2.set_ylim(-1.15, 1.15); ax2.set_xlim(-0.5, F-0.5); ax2.set_xticks(range(F))
    ax2.set_title(f"angular_z0  [min={ang.min():.3f}]", fontsize=8)
    ax2.set_xlabel("frame p")
    ax2.tick_params(labelsize=7)

    # pred_mag0 (velocity magnitude at final denoising step)
    ax3.bar(frames, f["pred_mag0"], width=0.75, color="#9C27B0", alpha=0.85)
    add_gt_vline(ax3, sid, gt_dict, label=False)
    shade_cond(ax3, F, F=F)
    ax3.set_xlim(-0.5, F-0.5); ax3.set_xticks(range(F))
    ax3.set_title("pred_mag0  ‖v_θ(p)‖  (τ=39)", fontsize=8)
    ax3.set_xlabel("frame p")
    ax3.tick_params(labelsize=7)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img_b64 = __import__("base64").b64encode(buf.read()).decode()

    vid_h  = video_html(r["video_path"], width=340, label=r["short_id"], cls=r["cls"])
    plot_h = f'<img src="data:image/png;base64,{img_b64}" style="height:195px;vertical-align:top">'
    all_html.append(
        f'<div style="display:flex;align-items:flex-start;gap:10px;'
        f'margin-bottom:10px;border-bottom:1px solid #ddd;padding-bottom:8px">'
        f'{vid_h}{plot_h}</div>'
    )

display(HTML("\n".join(all_html)))


---
## 1.4 — PCA of VAE Latent Frame Embeddings (`z₀`)

Complement to the bar-chart signals: project the 16 frame mean-embeddings of `z₀`
into 2D to see their geometric layout in one scatter.

**Computation:**  
For each latent frame `p`, average `z₀[C=128, :, :]` over the 16×24 spatial positions
→ one 128-dim vector. Stack all 16 → `[16, 128]` → centre → SVD → 2D.

| Symbol | Meaning |
|--------|---------|
| **Colour** | 🔵 blue=frame 0 → 🔴 red=frame 15 (temporal order) |
| **★** | Conditioned frame (`p=0..3`, `p=12..15`) — model must reconstruct exactly |
| **●** | Free-middle frame (`p=4..11`) — freely generated |
| **🟢 diamond** | GT annotation (if available) |
| **EV** | Explained variance of PC1 / PC2 |

**What to look for:**
- Conditioned frames (★) anchor opposite ends of the scatter; free-middle (●) interpolates.
- A **kink** in the frame trajectory = latent discontinuity at the transition.
- Compare shape across classes: does class 8 show a sharper kink than class 1?


In [ ]:
import gc

cmap_frames = plt.get_cmap("RdYlBu_r")
n_cols = 5
n_rows = (len(records) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.6, n_rows * 3.6))
axes_flat = axes.flatten()

fig.suptitle(
    "Level 1 — PCA of VAE Latent Frame Embeddings  (z₀, τ=39)\n"
    "Each dot = latent frame p.  Blue=early → Red=late.  "
    "★=conditioned  ●=free-middle  ◆=GT",
    fontsize=10, fontweight="bold"
)

for idx, r in enumerate(records):
    ax  = axes_flat[idx]
    sid = r["sample_id"]
    gp  = gt_latent(sid, gt_dict)

    print(f"  VAE PCA {r['short_id']} ...", end=" ", flush=True)
    _data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
    z0    = _data["z_final"].float()
    del _data; gc.collect()
    print("ok")

    frame_vecs  = z0.mean(dim=(2, 3)).T.numpy()   # [16, 128]
    frame_vecs -= frame_vecs.mean(axis=0)
    U, S_sv, _  = np.linalg.svd(frame_vecs, full_matrices=False)
    coords  = U[:, :2] * S_sv[:2]                # [16, 2]
    expvar  = (S_sv[:2]**2) / ((S_sv**2).sum() + 1e-12)

    # Draw temporal arrows
    for p in range(F_PRIME - 1):
        ax.annotate("", xy=(coords[p+1, 0], coords[p+1, 1]),
                    xytext=(coords[p, 0], coords[p, 1]),
                    arrowprops=dict(arrowstyle="-|>", color="gray", lw=0.5, alpha=0.3))

    for p in range(F_PRIME):
        c_val  = cmap_frames(p / (F_PRIME - 1))
        marker = "*" if (p < K_LAT or p >= END_IDX) else "o"
        ms     = 130 if marker == "*" else 65
        ax.scatter(coords[p, 0], coords[p, 1], color=c_val, s=ms,
                   marker=marker, zorder=3, alpha=0.9)
        ax.annotate(str(p), (coords[p, 0], coords[p, 1]),
                    fontsize=5, ha="center", va="center",
                    color="white" if marker == "o" else "k", fontweight="bold")

    # GT diamond
    if gp is not None:
        p_lo = int(gp); p_hi = min(p_lo + 1, F_PRIME - 1)
        frac = gp - p_lo
        gx   = coords[p_lo, 0] * (1 - frac) + coords[p_hi, 0] * frac
        gy   = coords[p_lo, 1] * (1 - frac) + coords[p_hi, 1] * frac
        ax.scatter(gx, gy, marker="D", s=80, color="#4CAF50", zorder=5,
                   edgecolors="black", linewidths=0.6, label=f"GT p≈{gp:.1f}")
        ax.legend(fontsize=5.5, loc="lower right", framealpha=0.7)

    ax.set_title(f"{r['short_id']}\nEV: {expvar[0]:.1%} / {expvar[1]:.1%}",
                 fontsize=7, color=r["color"])
    ax.set_xlabel("PC1", fontsize=6); ax.set_ylabel("PC2", fontsize=6)
    ax.tick_params(labelsize=5.5)

for ax in axes_flat[len(records):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()
